# SR-GB-CSNP — Colab Experiment Runner

Run the cells below in order:
1. **Setup** — mounts Drive, uploads/extracts your code ZIP, installs deps, smoke-tests MuJoCo.
2. **Full run** — runs `run_all_experiments.py` (no `--quick`). Takes several hours.

## Cell 1: Setup

In [ ]:
# ===================================================================
# SR‑GB‑CSNP – Colab Cell 1: Setup
# ===================================================================
# This is CELL 1 of 2. It will:
#   1. Mount Google Drive
#   2. Set up a working directory inside Drive
#   3. Upload and extract the code ZIP
#   4. Install Python dependencies (PySR/Julia auto-installs on first use)
#   5. Smoke-test MuJoCo
#
# Deliberately stops after setup rather than running experiments -- full
# stdout/stderr for every script are always saved to
# Results/logs/<script>.stdout.log / .stderr.log regardless. Once this
# cell completes cleanly, run the separate colab_run_full_cell.py in a
# new cell for the full (no --quick) suite.
#
# This file is not meant to be run with `python colab_setup_cell.py` --
# it uses Colab-only APIs (google.colab, get_ipython). Copy/paste its
# contents into a Colab notebook cell instead.

import os, sys, subprocess, time, shutil
from google.colab import drive, files

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
drive.mount('/content/drive')

# ------------------------------------------------------------
# 2. Set working directory inside Drive
# ------------------------------------------------------------
PROJECT_DIR = '/content/drive/MyDrive/sr_gb_csnp'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"📁 Working directory: {os.getcwd()}")

# ------------------------------------------------------------
# 3. Upload and extract the code ZIP
# ------------------------------------------------------------
print("📤 Please upload your project ZIP file (e.g., sr-gb-csnp.zip)")
uploaded = files.upload()
for fname in uploaded.keys():
    if not fname.endswith('.zip'):
        print(f"⚠️  Skipping non-ZIP file: {fname}")
        continue
    print(f"Unzipping {fname}...")
    get_ipython().system(f'unzip -o {fname} -d .')
    # Flatten: if a single top-level folder exists with .py files, move contents up
    extracted_items = os.listdir('.')
    for item in extracted_items:
        if os.path.isdir(item) and any(f.endswith('.py') for f in os.listdir(item)):
            for subitem in os.listdir(item):
                src = os.path.join(item, subitem)
                dst = os.path.join('.', subitem)
                if os.path.isdir(src):
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                    shutil.rmtree(src)
                else:
                    shutil.copy2(src, dst)
                    os.remove(src)
            os.rmdir(item)
            break
    os.remove(fname)
    print("✅ ZIP extracted and flattened (overwrote existing files).")

# ------------------------------------------------------------
# 4. Verify that key files are present
# ------------------------------------------------------------
print("\n📂 Project files (first 20 .py):")
get_ipython().system('ls -la *.py 2>/dev/null | head -20')

if not os.path.exists('run_all_experiments.py'):
    print("❌ run_all_experiments.py not found. Please check your ZIP upload.")
    sys.exit(1)

# ------------------------------------------------------------
# 5. Install Python dependencies
# ------------------------------------------------------------
# requirements.txt already pins everything needed: numpy/scipy/sympy/
# scikit-learn/pandas/matplotlib (core) + mujoco/z3-solver/pysr (optional,
# used by specific benchmarks). PySR's own juliacall dependency downloads
# and manages its own Julia install automatically on first use -- no
# separate system Julia install needed. Expect the *first* PySR-based
# benchmark call to be slow (one-time Julia download + precompile).
print("\n📦 Installing Python packages...")
get_ipython().system('pip install -q -r requirements.txt')
print("✅ Dependencies ready.")

# ------------------------------------------------------------
# 6. Smoke-test MuJoCo before committing to the full run
# ------------------------------------------------------------
# benchmark_robotics.py's model (pendulum.xml) and MuJoCo integration were
# fixed but never actually verified against a real MuJoCo install locally
# (mujoco crashes with an illegal-instruction error on that machine's CPU).
# This is the first real test of that fix -- check it here rather than
# discovering a problem after a multi-hour run.
print("\n🔍 Smoke-testing MuJoCo...")
mujoco_ok = subprocess.run(
    [sys.executable, "-c",
     "import mujoco; mujoco.MjModel.from_xml_path('pendulum.xml'); print('mujoco OK')"],
    capture_output=True, text=True
)
print(mujoco_ok.stdout.strip() or mujoco_ok.stderr.strip())

print("\n✅ Setup complete. Run colab_run_full_cell.py in a new cell for "
      "the full (no --quick) suite.")


## Cell 2: Full experiment suite (no `--quick`)

Run after Cell 1 completes.

In [ ]:
# ===================================================================
# SR‑GB‑CSNP – Colab Cell 2: Full experiment suite (no --quick)
# ===================================================================
# Run this in a separate cell AFTER colab_setup_cell.py completes. This
# is the multi-hour run.
#
# --generate-tables regenerates Results/all_paper_tables.tex from the
# freshly-written CSVs once every script has finished, so the LaTeX
# tables are ready to drop into the paper without a separate step.
#
# This file is not meant to be run with `python colab_run_full_cell.py` --
# it uses Colab-only APIs (get_ipython). Copy/paste its contents into a
# Colab notebook cell instead.

import os, sys, subprocess, time

PROJECT_DIR = '/content/drive/MyDrive/sr_gb_csnp'
os.chdir(PROJECT_DIR)

print("\n🚀 Starting full experiment suite (this will take several hours)...")
start_time = time.time()

process = subprocess.Popen([sys.executable, 'run_all_experiments.py', '--generate-tables'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT,
                           text=True,
                           bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()

elapsed = time.time() - start_time
print(f"\n⏱️ Total runtime: {elapsed/60:.1f} minutes")

# ------------------------------------------------------------
# Verify results are saved
# ------------------------------------------------------------
RESULTS_PATH = os.path.join(PROJECT_DIR, 'Results')
if os.path.exists(RESULTS_PATH):
    print(f"\n✅ Results saved to: {RESULTS_PATH}")
    get_ipython().system(f'ls -la {RESULTS_PATH}')
else:
    print("⚠️  Results folder not found. Check for errors above.")
